<a href="https://colab.research.google.com/github/manelsg22/microbiologie/blob/main/manelsg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

/content/Dataset on antimicrobial resistance of Vibrio spp. isolates across Adriatic regions (1).xlsx

### 1. Chargement et inspection des données
Commençons par lire le fichier Excel pour voir à quoi ressemblent les données, quelles sont les colonnes disponibles et identifier les types de variables.

In [2]:
import pandas as pd

# Chemin du fichier
file_path = '/content/Dataset on antimicrobial resistance of Vibrio spp. isolates across Adriatic regions (1).xlsx'

# Lire toutes les feuilles du fichier Excel pour ne rien manquer
excel_file = pd.ExcelFile(file_path)
print("Feuilles disponibles dans le fichier :", excel_file.sheet_names)

# Chargement de la première feuille par défaut
df = pd.read_excel(file_path, sheet_name=0)

# Affichage des premières lignes et des informations générales
display(df.head())
print("\n--- Informations sur le Dataset ---")
df.info()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Dataset on antimicrobial resistance of Vibrio spp. isolates across Adriatic regions (1).xlsx'

### Analyse et définition des variables $X$ et $Y$

Dans un problème d'analyse ou d'apprentissage automatique sur l'antibiorésistance de *Vibrio spp.*, nous séparons généralement les variables ainsi :
* **Variables Indépendantes ($X$)** : Les facteurs explicatifs (Région d'origine, espèce de *Vibrio*, type de matrice/milieu d'isolement, etc.).
* **Variables Cibles ($Y$)** : Les réponses à prédire ou expliquer (la sensibilité, la résistance intermédiaire ou totale aux différents antibiotiques testés).

In [ ]:
import pandas as pd

# Lecture du fichier en sautant les lignes d'en-tête inutiles si nécessaire
# Souvent dans ces tableaux, la première ligne (index 0) contient les vrais noms des colonnes
# Visualisons les premières lignes brutes pour localiser la bonne ligne d'en-tête
tmp_df = pd.read_excel(file_path, sheet_name='Final table')

# Affichons les 5 premières lignes brutes pour identifier où se trouvent les en-têtes
display(tmp_df.head(10))

In [1]:

*l
# En supposant que la ligne 0 ou 1 contient les vrais noms de colonnes,
# nous pouvons redéfinir les colonnes du DataFrame.
# Faisons un traitement dynamique pour nettoyer l'en-tête:

# Si la ligne d'index 0 contient les noms de colonnes (comme ID, Species, etc.) :
df_cleaned = tmp_df.copy()
df_cleaned.columns = df_cleaned.iloc[0]
df_cleaned = df_cleaned[1:].reset_index(drop=True)

print("Nouveaux noms de colonnes identifiés :")
print(df_cleaned.columns.tolist())

print("\n--- Aperçu des données nettoyées ---")
display(df_cleaned.head())

# Identification automatique des X potentiels et Y potentiels
# Les premières colonnes (ID, Species, Origin/Region) sont les variables explicatives (X)
# Les colonnes d'antibiotiques (ex: AMP, FOX, CIP, etc.) sont les cibles (Y)
col_list = df_cleaned.columns.tolist()

print("\n--- Proposition de répartition des variables ---")
print(f"Variables d'identification / caractéristiques (X potentiels) : {col_list[:5]}")
print(f"Profils de résistance aux antibiotiques (Y potentiels) : {col_list[5:]}")

NameError: name 'tmp_df' is not defined

### Nettoyage complet du Dataset
Nous allons maintenant procéder à un nettoyage rigoureux des données :
1. Correction de l'en-tête et suppression des premières lignes descriptives.
2. Suppression des espaces blancs invisibles au début ou à la fin des chaînes de caractères (ex: ` Vibrio` vs `Vibrio`).
3. Traitement des valeurs manquantes (`NaN`) de manière adaptée.
4. Vérification et standardisation des valeurs pour les antibiotiques (ex: s'assurer d'avoir uniquement 'R', 'I', 'S').

In [ ]:
import pandas as pd
import numpy as np

# 1. Charger la feuille brute
raw_df = pd.read_excel('/content/Dataset on antimicrobial resistance of Vibrio spp. isolates across Adriatic regions (1).xlsx', sheet_name='Final table')

# 2. Utiliser la ligne 0 comme noms de colonnes et supprimer les lignes inutiles au début
df_clean = raw_df.copy()
df_clean.columns = df_clean.iloc[0]
df_clean = df_clean[1:].reset_index(drop=True)

# 3. Supprimer les lignes entièrement vides si elles existent
df_clean = df_clean.dropna(how='all')

# 4. Nettoyer les espaces blancs dans toutes les colonnes textuelles (très important pour éviter les doublons)
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].astype(str).str.strip()
        # Rétablir les vrais NaN pour les cellules qui étaient vides
        df_clean[col] = df_clean[col].replace({'nan': np.nan, 'NA': np.nan, 'None': np.nan})

# 5. Remplir ou signaler les valeurs manquantes importantes
# Pour la région, si elle est manquante, on peut la remplacer par 'Unknown'
if 'Region' in df_clean.columns:
    df_clean['Region'] = df_clean['Region'].fillna('Unknown')

# 6. Affichage du résultat du nettoyage
print("Dimensions du dataset nettoyé :", df_clean.shape)
print("\nNombre de valeurs manquantes par colonne :")
print(df_clean.isnull().sum())

print("\nAperçu des données propres :")
display(df_clean.head())

### Sauvegarde du Dataset Nettoyé
Sauvegardons maintenant le résultat de ce nettoyage dans un nouveau fichier Excel indépendant.

In [ ]:
# Sauvegarde du DataFrame nettoyé dans un nouveau fichier Excel
output_path = '/content/Dataset_antimicrobial_resistance_cleaned.xlsx'
df_clean.to_excel(output_path, index=False)

print(f"Le jeu de données nettoyé a été sauvegardé avec succès à l'emplacement : {output_path}")

### 5. Division des données (Train/Test Split 75/25)
Pour évaluer l'efficacité de nos futurs modèles de prédiction, nous divisons notre jeu de données propre en :
* **75 %** pour la phase d'apprentissage (Entraînement).
* **25 %** gardés précieusement de côté pour l'évaluation finale (Test/Validation).

In [3]:
from sklearn.model_selection import train_test_split

# Nous définissons nos variables explicatives (X) et nos variables cibles (Y)
# Exclusion de l'I.D. et des variables non prédictives si nécessaire
X = df_clean[['Region', 'Origin of data', 'Sample type', 'Species']]
Y = df_clean[['ENR', 'FFC', 'GM', 'AM', 'E', 'T', 'SXT', 'UB', 'IPM', 'CIP', 'S', 'C']]

# Division 75% entraînement et 25% validation
# Nous stratifions sur l'espèce de Vibrio pour avoir une répartition équilibrée de chaque espèce dans les deux sets
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.25,
    random_state=42,
    stratify=X['Species']
)

# Affichage des dimensions de chaque ensemble de données
print(f"Dimensions du Dataset original  : {X.shape[0]} échantillons")
print(f"Dimensions du Set d'entraînement (75%) : {X_train.shape[0]} échantillons")
print(f"Dimensions du Set de validation   (25%) : {X_test.shape[0]} échantillons")

NameError: name 'df_clean' is not defined

### 2. Identification des variables liées (X et Y)
Une fois les colonnes identifiées, nous pouvons utiliser des méthodes statistiques (comme la corrélation pour les variables numériques, ou le test du Chi-deux pour les variables catégorielles de résistance) afin de détecter quelles variables $X$ sont statistiquement liées aux cibles $Y$ que vous souhaitez prédire.

In [ ]:
# Exemple générique d'analyse de corrélation pour les variables numériques
# Une fois que vous aurez exécuté la cellule précédente, vous pourrez ajuster la liste des X et Y.

# Sélection des colonnes numériques pour l'exemple
numeric_df = df.select_dtypes(include=['number'])

if not numeric_df.empty:
    print("Matrice de corrélation des variables numériques :")
    correlation_matrix = numeric_df.corr()
    display(correlation_matrix)
else:
    print("Pas de variables purement numériques ou besoin d'encoder des variables catégorielles.")

### 3. Analyse de la liaison entre les variables catégorielles (X et Y)
Puisque toutes les variables sont qualitatives (catégorielles), nous utilisons le **Test du Chi-deux d'indépendance** pour évaluer s'il existe une relation statistiquement significative entre les caractéristiques de l'isolat (X) et sa résistance aux antibiotiques (Y). Nous calculons également le **V de Cramér** pour mesurer la force de cette association (comprise entre 0 et 1).

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

# Définition des variables explicatives (X) et cibles (Y)
variables_x = ['Region', 'Origin of data', 'Sample type', 'Species']
variables_y = ['ENR', 'FFC', 'GM', 'AM', 'E', 'T', 'SXT', 'UB', 'IPM', 'CIP', 'S', 'C']

def cramers_v(x, y):
    """ Calcule le coefficient V de Cramér pour évaluer la force de l'association. """
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    # Ajustement pour éviter les biais
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

results = []

# Analyse croisée systématique
for x_col in variables_x:
    for y_col in variables_y:
        # On élimine les valeurs manquantes pour l'analyse
        temp_df = df_cleaned[[x_col, y_col]].dropna()
        if temp_df.empty or temp_df[x_col].nunique() <= 1 or temp_df[y_col].nunique() <= 1:
            continue

        contingency_table = pd.crosstab(temp_df[x_col], temp_df[y_col])
        chi2, p_value, _, _ = chi2_contingency(contingency_table)
        c_v = cramers_v(temp_df[x_col], temp_df[y_col])

        results.append({
            'Variable X': x_col,
            'Variable Y (Antibiotique)': y_col,
            'p-value (Chi2)': p_value,
            'V de Cramér (Force)': c_v,
            'Significatif (<0.05)': "Oui" if p_value < 0.05 else "Non"
        })

# Conversion en DataFrame pour affichage
analysis_df = pd.DataFrame(results)
significant_relations = analysis_df[analysis_df['Significatif (<0.05)'] == 'Oui'].sort_values(by='V de Cramér (Force)', ascending=False)

print("--- TOP DES LIAISONS LES PLUS FORTES (X -> Y) ---")
display(significant_relations.head(15))

### 4. Visualisation Graphique des Relations (X vs Y)
Créons des visualisations pour rendre l'analyse statistique beaucoup plus parlante et facile à comprendre.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Préparation de la matrice du V de Cramér pour la carte de chaleur
pivot_df = analysis_df.pivot(index='Variable X', columns='Variable Y (Antibiotique)', values='V de Cramér (Force)')

# Tri des colonnes et lignes pour une meilleure lecture
pivot_df = pivot_df.reindex(index=variables_x, columns=variables_y)

# Tracé de la Heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(pivot_df, annot=True, cmap='YlOrRd', fmt=".2f", linewidths=0.5, cbar_kws={'label': "Force de l'association (V de Cramér)"})
plt.title("Intensité de l'influence des variables X sur la résistance aux antibiotiques (Y)", fontsize=14, pad=15)
plt.xlabel("Antibiotiques (Variables Y)", fontsize=12)
plt.ylabel("Caractéristiques de l'isolat (Variables X)", fontsize=12)
plt.tight_layout()
plt.show()

In [4]:
# 2. Zoom sur la relation la plus forte : Région vs Tétracycline (T)
# Nettoyage rapide pour ne garder que les données non nulles pertinentes pour le graphique
plot_data = df_cleaned[['Region', 'T']].dropna()

# Création d'un tableau croisé en pourcentages pour comparer équitablement les régions
cross_tab = pd.crosstab(plot_data['Region'], plot_data['T'], normalize='index') * 100

# Tracé des barres empilées
cross_tab.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#ff9999','#ffff99','#99ff99'])
plt.title("Distribution des profils de résistance à la Tétracycline (T) par Région", fontsize=14, pad=15)
plt.xlabel("Région (X)", fontsize=12)
plt.ylabel("Pourcentage (%)", fontsize=12)
plt.legend(title="Profil (Y)", labels=['Résistant (R)', 'Intermédiaire (I)', 'Sensible (S)'], bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

NameError: name 'df_cleaned' is not defined